# 🛰️ GeoFM: Dataset Builder & Chip Packager (Google Colab)
### *High-Resolution Bitemporal Sentinel-2 & Hansen GFC Ingestion Suite*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safiydev/geofm-deforestation/blob/main/notebooks/geofm_dataset_builder.ipynb)
![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)
![Python](https://img.shields.io/badge/Python-3.9%20%7C%203.10%20%7C%203.11-brightgreen)

This standalone Colab notebook allows you to **fetch, validate, and package bitemporal satellite datasets** into a standardized `geofm_dataset.zip` archive.

### 🌟 Why Package Datasets into a ZIP?
1. **Zero Repeated Downloads**: Fetching hundreds of cloud-free Sentinel-2 multi-spectral tiles and pairing them with 30m Hansen Global Forest Change ground-truth masks takes 10–25 minutes. Once packaged into a `.zip`, you can load it in **~3 seconds** on subsequent training runs or share it with your team.
2. **Plug-and-Play Training Input**: The generated `.zip` can be uploaded directly into the **GeoFM Training Notebook** (`geofm_training.ipynb`), the **Kaggle GPU Pipeline**, the **CLI Trainer** (`geofm/train_cli.py`), or the **Model Loader Studio** (`model_loader.py`).
3. **Multi-Biome Coverage**: Sample real deforestation hot-spots across the **Amazon Arc of Deforestation**, **Congo Basin**, **Southeast Asia (Borneo/Sumatra)**, or **Gran Chaco** with enforced 70:30 class balance.

---
## 1. Environment Setup & Dependency Installation
Clones the GeoFM repository and installs lightweight geospatial parsing libraries (earthengine-api, rasterio, tifffile, pystac).

In [ ]:
# 1. Environment Setup
import os
import sys
import shutil
from pathlib import Path

WORKING_DIR = "/content" if os.path.exists("/content") else os.getcwd()
REPO_DIR = os.path.join(WORKING_DIR, "geofm-deforestation")

if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
    print(f"✓ Cloning GeoFM from GitHub...")
    !git clone https://github.com/safiydev/geofm-deforestation.git {REPO_DIR}

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install geospatial parsing dependencies
!pip install -q earthengine-api rasterio tifffile pillow pystac requests numpy
!pip install -q -e .

print("✓ GeoFM Dataset Builder Environment Ready!")

---
## 2. Interactive Dataset Configuration
Choose your data source, target chip count, spatial resolution, and class balancing quota.

> ### 🔑 Google Earth Engine Note:
> - Enter your Google Cloud Project ID in `GEE_PROJECT_ID` (e.g. `ee-yourname`). If you leave it blank, the system automatically uses public streaming or Hugging Face forest loss imagery.

In [ ]:
# @title 🛠️ Dataset Builder Parameters { run: 'auto' }

DATASET_SOURCE = 'Google Earth Engine + Hansen GFC (Global Biomes)' #@param ['Google Earth Engine + Hansen GFC (Global Biomes)', 'Real Forest Loss Imagery (JimmyBrocko/Forest-Change HF)', 'Real Sentinel-2 Satellite Imagery (OSCD Benchmark)', 'Synthetic Satellite Simulation (Offline / Fast)']
SCALE_OPTION = 'Production Baseline (500 chips)' #@param ['Quick Smoke Test (30 chips)', 'Regional Pilot (100 chips)', 'Production Baseline (500 chips)', 'Benchmark Scale (1000 chips)', 'Custom Chip Count']
CUSTOM_CHIP_COUNT = 250 #@param {type:'integer'}
CHIP_SIZE = 256 #@param [128, 256, 512] {type:'raw'}
BIOME_FOCUS = 'All Global Tropical Biomes' #@param ['All Global Tropical Biomes', 'Amazon Arc of Deforestation (Brazil/Bolivia)', 'Congo Basin (DRC/Gabon)', 'Southeast Asia (Indonesia/Malaysia)', 'Cerrado / Gran Chaco']
ENFORCE_CLASS_BALANCE = True #@param {type:'boolean'} # 70% true deforestation, 30% intact canopy hard negatives
TARGET_POSITIVE_RATIO = 0.70 #@param {type:'number'}
PURGE_EXISTING_CACHE = True #@param {type:'boolean'}
ZIP_FILENAME = 'geofm_dataset.zip' #@param {type:'string'}
# GEE Project ID (Leave blank to use default or non-GEE fallback)
GEE_PROJECT_ID = '' #@param {type:'string'}

# Map scale selection to integer chip counts
scale_map = {
    'Quick Smoke Test (30 chips)': 30,
    'Regional Pilot (100 chips)': 100,
    'Production Baseline (500 chips)': 500,
    'Benchmark Scale (1000 chips)': 1000,
    'Custom Chip Count': CUSTOM_CHIP_COUNT,
}
NUM_CHIPS = scale_map.get(SCALE_OPTION, 500)

print("✓ Target Dataset Configuration:")
print(f"  • Data Source:       {DATASET_SOURCE}")
print(f"  • Target Chips:      {NUM_CHIPS} paired scenes ({CHIP_SIZE}x{CHIP_SIZE} px)")
print(f"  • Biome Focus:       {BIOME_FOCUS}")
print(f"  • Class Balance:     {ENFORCE_CLASS_BALANCE} ({TARGET_POSITIVE_RATIO*100:.0f}% positive / {(1-TARGET_POSITIVE_RATIO)*100:.0f}% hard negative)")
print(f"  • Output ZIP Name:   {ZIP_FILENAME}")

---
## 3. Planetary Satellite Data Ingestion
Downloads and standardizes bitemporal satellite rasters ($T_1$ baseline and $T_2$ observation) along with binary forest disturbance ground-truth masks.

In [ ]:
import os
import shutil
from pathlib import Path

cache_dir = "./data/chips_cache"
if PURGE_EXISTING_CACHE and os.path.exists(cache_dir):
    print(f"🧹 Purging old cache directory: {cache_dir}...")
    shutil.rmtree(cache_dir, ignore_errors=True)
os.makedirs(cache_dir, exist_ok=True)

# Execute Ingestion according to chosen source
if DATASET_SOURCE.startswith("Google Earth Engine"):
    print(f"🛰️ Streaming {NUM_CHIPS} multi-spectral Sentinel-2 + Hansen GFC chips from Google Earth Engine...")
    from geofm.data_sources.earth_engine import download_and_cache_gee_dataset
    manifest_path, total_cached = download_and_cache_gee_dataset(
        output_dir=cache_dir,
        max_chips=NUM_CHIPS,
        project_id=GEE_PROJECT_ID.strip() if GEE_PROJECT_ID.strip() else None,
        chip_size=CHIP_SIZE,
        class_balance_filter=ENFORCE_CLASS_BALANCE,
        target_pos_ratio=TARGET_POSITIVE_RATIO,
        max_workers=8,
        overwrite_cache=PURGE_EXISTING_CACHE,
        show_progress=True,
    )
elif DATASET_SOURCE.startswith("Real Forest"):
    print(f"🛰️ Ingesting JimmyBrocko/Forest-Change Deforestation Dataset from Hugging Face...")
    from geofm.utils.real_data import download_and_cache_forest_change_dataset
    manifest_path, total_cached = download_and_cache_forest_change_dataset(
        output_dir=cache_dir,
        max_chips=NUM_CHIPS,
        chip_size=CHIP_SIZE,
        class_balance_filter=ENFORCE_CLASS_BALANCE,
        target_pos_ratio=TARGET_POSITIVE_RATIO,
        show_progress=True,
    )
elif DATASET_SOURCE.startswith("Real Sentinel"):
    print(f"🛰️ Ingesting Real Sentinel-2 Multi-Spectral Imagery from ESA / Hugging Face...")
    from geofm.utils.real_data import download_and_cache_real_sentinel2_dataset
    manifest_path, total_cached = download_and_cache_real_sentinel2_dataset(
        output_dir=cache_dir,
        max_chips=NUM_CHIPS,
        show_progress=True,
    )
else:
    print(f"🛰️ Generating {NUM_CHIPS} synthetic multi-spectral bitemporal satellite chips...")
    from geofm.utils.synthetic_data import stream_and_cache_synthetic_dataset
    manifest_path, total_cached = stream_and_cache_synthetic_dataset(
        output_dir=cache_dir,
        num_samples=NUM_CHIPS,
        height=CHIP_SIZE,
        width=CHIP_SIZE,
        num_bands=6,
        append_indices=False,
        positive_ratio=TARGET_POSITIVE_RATIO if ENFORCE_CLASS_BALANCE else 0.50,
        coord_step=0.04,
        batch_flush=50,
        show_progress=True,
    )

cached_files = list(Path(cache_dir).glob("*.npz"))
print(f"\n✓ Successfully cached {len(cached_files)} serialized chips in: {cache_dir}")

---
## 4. Visual Quality Control & Spectral Inspection
Inspect randomly selected chips from the dataset to confirm cloud-free quality, spectral fidelity, and pixel-aligned ground truth deforestation masks.

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

chip_files = sorted(glob.glob(f"{cache_dir}/*.npz"))
if not chip_files:
    print("⚠️ No chips found in cache to visualize.")
else:
    # Select 4 representative chips
    step = max(1, len(chip_files) // 4)
    sample_picks = chip_files[::step][:4]
    
    fig, axes = plt.subplots(len(sample_picks), 4, figsize=(16, 4 * len(sample_picks)))
    if len(sample_picks) == 1:
        axes = np.expand_dims(axes, 0)

    for i, c_path in enumerate(sample_picks):
        data = np.load(c_path)
        t1 = data['t1_optical']
        t2 = data['t2_optical']
        mask = data.get('target_mask', np.zeros((1, t1.shape[1], t1.shape[2])))
        sample_id = str(data.get('sample_id', Path(c_path).stem))
        loss_pct = float(data.get('deforested_pct', 0.0))

        # RGB normalization helper (percentile stretch 2% - 98%)
        def to_rgb(arr):
            if arr.ndim == 3 and arr.shape[0] >= 3:
                rgb = arr[:3].transpose(1, 2, 0)
            else:
                rgb = np.repeat(arr[0:1], 3, axis=0).transpose(1, 2, 0)
            p2, p98 = np.percentile(rgb, (2, 98))
            return np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0.0, 1.0)

        t1_rgb = to_rgb(t1)
        t2_rgb = to_rgb(t2)
        diff = np.abs(t2_rgb - t1_rgb).mean(axis=-1)

        # 1. T1 Baseline
        axes[i, 0].imshow(t1_rgb)
        axes[i, 0].set_title(f"[{sample_id}] T1 Pre-Change", fontsize=10, fontweight='bold')
        axes[i, 0].axis('off')

        # 2. T2 Observation
        axes[i, 1].imshow(t2_rgb)
        axes[i, 1].set_title(f"T2 Post-Change (Loss: {loss_pct:.1f}%)", fontsize=10, fontweight='bold')
        axes[i, 1].axis('off')

        # 3. Spectral Delta
        im_diff = axes[i, 2].imshow(diff, cmap='inferno')
        axes[i, 2].set_title("Spectral Delta |T2 - T1|", fontsize=10, fontweight='bold')
        axes[i, 2].axis('off')

        # 4. Ground Truth Deforestation Mask
        m = mask.squeeze()
        axes[i, 3].imshow(t2_rgb)
        axes[i, 3].imshow(np.ma.masked_where(m == 0, m), cmap='Reds', alpha=0.6)
        axes[i, 3].set_title("Ground Truth Forest Loss", fontsize=10, fontweight='bold')
        axes[i, 3].axis('off')

    plt.suptitle("GeoFM Dataset Quality Control Verification", fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

---
## 5. Dataset Metrics & Class Balance Summary
Computes statistical properties across the entire dataset to ensure zero class collapse during training.

In [ ]:
import glob
import numpy as np

chip_files = glob.glob(f"{cache_dir}/*.npz")
total_chips = len(chip_files)
positive_chips = 0
negative_chips = 0
total_deforested_pixels = 0
total_pixels = 0

for f in chip_files:
    data = np.load(f)
    m = data.get('target_mask', np.zeros((1, 256, 256)))
    def_pixels = int(np.sum(m > 0))
    tot = m.size
    pct = (def_pixels / tot) * 100.0
    
    if pct >= 5.0:
        positive_chips += 1
    else:
        negative_chips += 1
        
    total_deforested_pixels += def_pixels
    total_pixels += tot

# 10m Sentinel-2 pixel = 0.01 hectares
total_area_ha = total_deforested_pixels * 0.01

print("=" * 60)
print("📊 GeoFM Dataset Inventory & Balance Report")
print("=" * 60)
print(f"  • Total Paired Chips:       {total_chips}")
print(f"  • Positive Chips (>5% loss): {positive_chips} ({(positive_chips/total_chips)*100:.1f}%)")
print(f"  • Negative / Intact Chips:  {negative_chips} ({(negative_chips/total_chips)*100:.1f}%)")
print(f"  • Total Deforested Area:    {total_area_ha:,.2f} hectares ({total_area_ha/100:,.2f} km²)")
print(f"  • Landscape Loss Density:   {(total_deforested_pixels / total_pixels)*100:.2f}% of monitored pixels")
print("=" * 60)

---
## 6. Package into ZIP Archive & Download
Compresses `./data/chips_cache` into the final dataset archive and triggers automatic browser download.

In [ ]:
import os
import shutil
from pathlib import Path

zip_base = os.path.splitext(ZIP_FILENAME)[0]
output_zip_path = os.path.join(WORKING_DIR, ZIP_FILENAME)

print(f"📦 Compressing {len(chip_files)} chips into {ZIP_FILENAME}...")
# Archive data/chips_cache
shutil.make_archive(os.path.join(WORKING_DIR, zip_base), 'zip', cache_dir)

file_size_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
print(f"✓ Package created successfully!")
print(f"  • Path:      {output_zip_path}")
print(f"  • File Size: {file_size_mb:.2f} MB")

# Trigger Colab automated file download
try:
    from google.colab import files
    print("⚡ Initiating browser download...")
    files.download(output_zip_path)
    print("✓ Download started! Check your browser's download manager.")
except ImportError:
    print(f"ℹ️ File saved to local disk: {output_zip_path}")

---
## 7. How to Use this ZIP Archive in GeoFM

### A. In Google Colab Training Notebook (`geofm_training.ipynb`):
1. Open [`geofm_training.ipynb`](https://colab.research.google.com/github/safiydev/geofm-deforestation/blob/main/notebooks/geofm_training.ipynb).
2. Drag and drop your downloaded `geofm_dataset.zip` into the Colab Files sidebar on the left.
3. The training pipeline automatically detects and unpacks your pre-generated dataset in **~2 seconds**, bypassing data collection entirely!

### B. In the Standalone Model Loader / Web Studio:
1. Unzip `geofm_dataset.zip` into `./data/chips_cache` on your local machine.
2. Run:
   ```bash
   python model_loader.py --port 8600
   ```
3. Open `http://localhost:8600` and immediately test any of the packaged chips from the dropdown menu!

### C. In the CLI Training Engine:
```bash
unzip geofm_dataset.zip -d ./data/my_dataset
python geofm/train_cli.py --data-dir ./data/my_dataset --epochs 30 --batch-size 16
```